# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ayesha-Shahzadkhan/flyrank-assignment1/blob/main/work/notebooks/w03_data_contract.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**Unit of analysis:**
One row = one content piece (article/page), identified by `content_hash_id`, from the `dim_content` table.

**Time window:**
Content pieces created on or before March 2026 (`content_created_date <= 2026-03-31`). This is used as the mid-panel month for development; the final month (June 2026, the `_sample` table) is kept as a sealed test month.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

**Feature** (used to build the clustering features):
- `char_count`, `word_count`, `category_count` — describe content size/shape
- `content_type` — describes format type
- `provider_used` / `model_used` — describes generation style/format

**Label / Proxy**:
No strict label — this is unsupervised clustering. `content_type` can be used
as a rough proxy afterward, only to sanity-check whether clusters roughly
align with known content formats — not used as a training input.

**Context** (useful for understanding, not used as a feature):
`content_hash_id`, `client_hash_id`, `content_created_date`,
`content_updated_date`, `is_published`, `is_deleted`

**Excluded**:
- `search_volume`, `competition`, `competition_level`, `cpc`, `main_intent`,
  `backlinks` — keyword/SEO performance metrics, not structural properties
  of the content itself; irrelevant to structure-based clustering.
- `keyword_hash_id`, `keyword_char_count`, `keyword_token_count`,
  `url_hash_id`, `url_char_count`, `keyword_created_date` — describe the
  associated keyword/URL, not the content's own structure.
- `last_optimized_date`, `optimization_eligible_date` — workflow/scheduling
  dates, not structural signals.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [3]:
from google.colab import userdata
import pandas as pd

hf_token = userdata.get('HF_TOKEN')

df_content = pd.read_parquet(
    "hf://datasets/FlyRank/internship-warehouse/dim_content.parquet",
    storage_options={"token": hf_token}
)

print(df_content.shape)
df_content.head()

(519606, 26)


,client_hash_id,content_hash_id,keyword_hash_id,url_hash_id,keyword_char_count,keyword_token_count,url_char_count,content_created_date,content_updated_date,content_type,...,category_count,keyword_created_date,provider_used,model_used,char_count,word_count,last_optimized_date,optimization_eligible_date,is_published,is_deleted
0,client_04660893ae39614a,content_004de9653278b5a4,keyword_e754999ab88dd9f2,url_d6091f18cf628794,22,4,108,2026-05-30,2026-07-01,keyword article,...,3,2026-05-12,gemini-generate-content,gemini-3-flash-preview,15682.0,2555.0,None,None,True,False
1,client_04660893ae39614a,content_00dc5efae381b2ab,keyword_4329d7aede8e208b,url_3a66d2f2e36823ca,31,6,95,2026-06-12,2026-07-01,keyword article,...,4,2026-06-01,gemini-generate-content,gemini-3-flash-preview,15438.0,2430.0,None,None,True,False
2,client_04660893ae39614a,content_01410f2556c327ac,keyword_9b08047d3d2a0406,url_809eda7a7e20b3b2,22,5,82,2026-05-09,2026-07-01,keyword article,...,4,2026-05-06,gemini-generate-content,gemini-3-flash-preview,16576.0,2645.0,None,None,True,False
3,client_04660893ae39614a,content_019f27f634053ca7,keyword_e7cec7ab1804c1c2,url_5fb42bafc4399861,14,3,92,2026-06-15,2026-06-15,keyword article,...,4,2026-06-01,gemini-generate-content,gemini-3-flash-preview,15457.0,2522.0,None,None,True,False
4,client_04660893ae39614a,content_01efa71faea45dcc,keyword_56b0062a1d8b7524,url_ece0abc3e5fb75f9,24,6,98,2026-05-21,2026-06-01,keyword article,...,4,2026-05-12,gemini-generate-content,gemini-3-flash-preview,15776.0,2552.0,None,None,True,False


In [4]:
# Query 1: Grain check — does content_hash_id uniquely identify each row?
total_rows = len(df_content)
unique_content_ids = df_content['content_hash_id'].nunique()

print(f"Total rows: {total_rows}")
print(f"Unique content_hash_id: {unique_content_ids}")
print(f"Grain confirmed (1 row = 1 content piece): {total_rows == unique_content_ids}")

Total rows: 519606
Unique content_hash_id: 519606
Grain confirmed (1 row = 1 content piece): True


In [5]:
import datetime

# Query 2: Row count + date span for the mid-panel month slice (up to March 2026)
cutoff = datetime.date(2026, 3, 31)
df_march = df_content[df_content['content_created_date'] <= cutoff]

print(f"Rows created on or before March 2026: {len(df_march)}")
print(f"Earliest content_created_date: {df_march['content_created_date'].min()}")
print(f"Latest content_created_date: {df_march['content_created_date'].max()}")

Rows created on or before March 2026: 433434
Earliest content_created_date: 2024-10-16
Latest content_created_date: 2026-03-31


In [6]:
# Query 3: Availability check — filter to published, non-deleted content (within March slice)
before_count = len(df_march)
df_available = df_march[(df_march['is_published'] == True) & (df_march['is_deleted'] == False)]
after_count = len(df_available)

print(f"Rows before availability filter: {before_count}")
print(f"Rows after filter (is_published IS TRUE, is_deleted IS FALSE): {after_count}")
print(f"Rows dropped: {before_count - after_count}")

Rows before availability filter: 433434
Rows after filter (is_published IS TRUE, is_deleted IS FALSE): 329534
Rows dropped: 103900


In [7]:
# Build the 5-feature frame for structure/format clustering
features = df_available[[
    'content_hash_id',
    'char_count',
    'word_count',
    'category_count',
    'content_type',
    'provider_used'
]].copy()

print(features.shape)
features.head()

(329534, 6)


,content_hash_id,char_count,word_count,category_count,content_type,provider_used
6877,content_004e9c4c32e88631,24997.0,3935.0,6,keyword article,None
6878,content_0236ef736698e17c,27584.0,4335.0,2,keyword article,None
6879,content_025f6cfd3c298870,22998.0,3719.0,6,keyword article,None
6880,content_0263d5f9b7a2ecd4,19942.0,3246.0,0,keyword article,None
6881,content_02752c6c1c60161f,22584.0,3641.0,4,keyword article,None


In [8]:
features['provider_used'].isna().sum()

np.int64(267828)

In [9]:
df_available['model_used'].isna().sum()

np.int64(83833)

In [10]:
features = df_available[[
    'content_hash_id',
    'char_count',
    'word_count',
    'category_count',
    'content_type',
    'model_used'
]].copy()

print(features.shape)
features.head()

(329534, 6)


,content_hash_id,char_count,word_count,category_count,content_type,model_used
6877,content_004e9c4c32e88631,24997.0,3935.0,6,keyword article,gemini-2.5-flash
6878,content_0236ef736698e17c,27584.0,4335.0,2,keyword article,gemini-2.5-flash
6879,content_025f6cfd3c298870,22998.0,3719.0,6,keyword article,gemini-2.5-flash
6880,content_0263d5f9b7a2ecd4,19942.0,3246.0,0,keyword article,gemini-2.5-flash
6881,content_02752c6c1c60161f,22584.0,3641.0,4,keyword article,gemini-2.5-flash


**Feature availability at decision moment:**

- `char_count` — known as soon as the content is created/published; no future data needed.
- `word_count` — same as char_count, available at creation time.
- `category_count` — assigned when the content is categorized, which happens at creation.
- `content_type` — decided when the content is created (part of its design).
- `model_used` — recorded automatically at generation time, since it's the AI model that produced the content.


**Deliberate leakage experiment:**

We added a column (`leaky_content_type`) derived directly from `content_type` —
our proxy/label — scaled to dominate the distance calculation.

- Honest silhouette score (5 real features): 0.676
- Silhouette score WITH the leak: 0.921 (misleadingly near-perfect)
- After removing the leak: 0.676 (back to the honest number)

This shows how a single label-derived feature can make clustering look
almost perfect while actually just re-encoding the answer — not learning
real structure. The leaky column was removed from the final feature set.


In [20]:
from sklearn.preprocessing import LabelEncoder
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# Clean rebuild — drop missing values first
features_clean = features.dropna()

# Encode the two categorical features
le_type = LabelEncoder()
le_model = LabelEncoder()

X = pd.DataFrame()
X['char_count'] = features_clean['char_count']
X['word_count'] = features_clean['word_count']
X['category_count'] = features_clean['category_count']
X['content_type_enc'] = le_type.fit_transform(features_clean['content_type'])
X['model_used_enc'] = le_model.fit_transform(features_clean['model_used'])

print(X.columns.tolist())
print(X.shape)

['char_count', 'word_count', 'category_count', 'content_type_enc', 'model_used_enc']
(222744, 5)


In [21]:
kmeans_honest = KMeans(n_clusters=4, random_state=42, n_init=10)
labels_honest = kmeans_honest.fit_predict(X)

score_honest = silhouette_score(X, labels_honest, sample_size=5000, random_state=42)
print(f"Silhouette Score (Honest): {score_honest:.3f}")

Silhouette Score (Honest): 0.676


In [24]:
X_leaky = X.copy()
X_leaky['leaky_content_type'] = X_leaky['content_type_enc'] * 100000

kmeans_leaky = KMeans(n_clusters=3, random_state=42, n_init=10)
Labels_leaky = kmeans_leaky.fit_predict(X_leaky)

score_leaky = silhouette_score(X_leaky, Labels_leaky, sample_size=5000, random_state=42)
print(f"Silhouette Score (Leaky): {score_leaky:.3f}")

Silhouette Score (Leaky): 0.921


In [25]:
# Remove the leaky column — go back to honest features
X_final = X.drop(columns=['leaky_content_type']) if 'leaky_content_type' in X.columns else X

kmeans_final = KMeans(n_clusters=4, random_state=42, n_init=10)
labels_final = kmeans_final.fit_predict(X_final)

score_final = silhouette_score(X_final, labels_final, sample_size=5000, random_state=42)
print(f"Silhouette Score (Honest, leak removed): {score_final:.3f}")

Silhouette Score (Honest, leak removed): 0.676


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

**Named limitation of this slice:**

Our feature set relies on `model_used`, which is missing for about 25% of
rows (and `provider_used` was missing for 81%, so it was dropped entirely).
This means AI-generation metadata is inconsistently recorded across content
pieces — likely because not all content in this warehouse was generated the
same way, or logging started partway through. Any cluster that leans on
`model_used` may be biased toward content where this field happens to be
recorded, and doesn't tell us anything about the structure of content where
it's missing. This is a gap in the data itself, not something we can fix
with more features.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.